# MMS adapter fine-tuning
based on https://huggingface.co/blog/mms_adapters

In [1]:
%pip install -U pip
%pip install --no-cache-dir "transformers==4.57.1" accelerate "datasets[audio]" evaluate jiwer safetensors huggingface_hub tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 55.5 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 4.3 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 8.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 9.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 12.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 12.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.6 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 30.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 30.9 MB/s  0:00:00e

In [2]:
from huggingface_hub import (login, notebook_login, create_repo, HfFolder, snapshot_download)

import torch
from datasets import load_dataset, Audio

import json
from pathlib import Path
import os
import gc
import shutil
import re
import time

from transformers import (
    Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor,
    Wav2Vec2ForCTC, TrainingArguments, Trainer, EarlyStoppingCallback
)

from dataclasses import dataclass
from typing import Dict, List, Union

import numpy as np
import jiwer
import pandas as pd

from safetensors.torch import save_file as safe_save_file
from transformers.models.wav2vec2.modeling_wav2vec2 import WAV2VEC2_ADAPTER_SAFE_FILE

from tqdm.auto import tqdm

In [3]:
model_id = 'facebook/mms-1b-all'
target_lang = 'ckt'
dataset_repo_id = 'tadgeis/chukchi-asr-data-private'

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

True
NVIDIA A100-SXM4-80GB


In [4]:
notebook_login()

In [5]:
hf_token = HfFolder.get_token()

if hf_token is None:
    raise ValueError('HF token was not found.')

In [6]:
dataset_dict = load_dataset(dataset_repo_id, token=hf_token)
dataset_dict = dataset_dict.cast_column('audio', Audio(sampling_rate=16_000))

dataset_dict

README.md:   0%|          | 0.00/422 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/411M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/410M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/145M [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/60.0M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating dev split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 2714
    })
    test: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 429
    })
    dev: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 140
    })
})

In [7]:
example = dataset_dict['train'][1000]

print(example.keys())
print(example['resource'])
print(example['path'])
print(example['sentence'])
print(example['audio']['sampling_rate'])
print(example['audio']['array'].shape)

dict_keys(['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'])
chuklang
I am from Chukotka_5.wav
мури вай амваанвыкэнайгым гым
16000
(42240,)


In [8]:
vocab_source_dataset = dataset_dict['train']

def extract_all_chars(batch):
    all_text = ' '.join(batch['sentence'])
    vocab = sorted(list(set(all_text)))
    return {'vocab': [vocab], 'all_text': [all_text]}

vocab_result = vocab_source_dataset.map(extract_all_chars, batched=True,
    batch_size=-1, keep_in_memory=True,
    remove_columns=vocab_source_dataset.column_names)

vocab_list = vocab_result['vocab'][0]

print(vocab_list)
print('Number of raw characters:', len(vocab_list))

Map:   0%|          | 0/2714 [00:00<?, ? examples/s]

[' ', "'", 'а', 'б', 'в', 'г', 'д', 'е', 'ж', 'з', 'и', 'й', 'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'у', 'ф', 'х', 'ц', 'ч', 'ш', 'щ', 'ъ', 'ы', 'ь', 'э', 'ю', 'я', 'ё', 'ӄ', 'ӈ', 'ԓ']
Number of raw characters: 38


In [9]:
vocab_dict = {char: idx for idx, char in enumerate(vocab_list)}

if ' ' not in vocab_dict:
    raise ValueError('Space character is not in vocabulary.')

vocab_dict['|'] = vocab_dict[' ']
del vocab_dict[' ']

vocab_dict['[UNK]'] = len(vocab_dict)
vocab_dict['[PAD]'] = len(vocab_dict)

In [10]:
new_vocab_dict = {target_lang: vocab_dict}

with open('vocab.json', 'w', encoding='utf-8') as vocab_file:
    json.dump(new_vocab_dict, vocab_file, ensure_ascii=False, indent=2)

In [11]:
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained("./", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|", target_lang=target_lang)

In [12]:
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16_000,
    padding_value=0.0, do_normalize=True, return_attention_mask=True)

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

In [13]:
def prepare_dataset(batch):

    audio = batch['audio']

    batch['input_values'] = processor(audio['array'], sampling_rate=audio['sampling_rate']).input_values[0]
    batch['input_length'] = len(batch['input_values'])

    batch['labels'] = processor(text=batch['sentence']).input_ids
    return batch


@dataclass
class DataCollatorCTCWithPadding:
    """
    Data collator that will dynamically pad the inputs received.
    Args:
        processor (:class:`~transformers.Wav2Vec2Processor`)
            The processor used for proccessing the data.
        padding (:obj:`bool`, :obj:`str` or :class:`~transformers.tokenization_utils_base.PaddingStrategy`, `optional`, defaults to :obj:`True`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding index)
            among:
            * :obj:`True` or :obj:`'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
              sequence if provided).
            * :obj:`'max_length'`: Pad to a maximum length specified with the argument :obj:`max_length` or to the
              maximum acceptable input length for the model if that argument is not provided.
            * :obj:`False` or :obj:`'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of
              different lengths).
    """

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need
        # different padding methods
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels

        return batch


def normalize_spaces(text):
    if pd.isna(text):
        return ''

    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    # we do not want to group tokens when computing the metrics
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    pred_str = [normalize_spaces(text) for text in pred_str]
    label_str = [normalize_spaces(text) for text in label_str]

    wer = jiwer.wer(label_str, pred_str)
    cer = jiwer.cer(label_str, pred_str)

    return {'wer': wer, 'cer': cer}


def create_mms_adapter_model():
    model = Wav2Vec2ForCTC.from_pretrained(
        'facebook/mms-1b-all',
        attention_dropout=0.0,
        hidden_dropout=0.0,
        feat_proj_dropout=0.0,
        layerdrop=0.0,
        ctc_loss_reduction='mean',
        pad_token_id=processor.tokenizer.pad_token_id,
        vocab_size=len(processor.tokenizer),
        ignore_mismatched_sizes=True,
    )

    model.init_adapter_layers()
    model.freeze_base_model()

    adapter_weights = model._get_adapters()

    for param in adapter_weights.values():
        param.requires_grad = True

    return model


def load_trained_mms_adapter_model(model_path):
    model = Wav2Vec2ForCTC.from_pretrained(model_path)

    model.freeze_base_model()

    adapter_weights = model._get_adapters()

    for param in adapter_weights.values():
        param.requires_grad = True

    return model


def prepare_resource_test_dataset(resource):
    test_dataset_raw_resource = dataset_dict['test'].filter(
        lambda example: example['resource'] == resource
    )

    if len(test_dataset_raw_resource) == 0:
        raise ValueError(f'No test examples found for resource: {resource}')

    test_dataset_resource = test_dataset_raw_resource.map(
        prepare_dataset,
        remove_columns=test_dataset_raw_resource.column_names,
        load_from_cache_file=False
    )

    decoded_labels = processor.batch_decode(test_dataset_resource['labels'], group_tokens=False)
    decoded_labels = [normalize_spaces(text) for text in decoded_labels]

    references = [normalize_spaces(text) for text in test_dataset_raw_resource['sentence']]
    mismatches = [
        (i, ref, label)
        for i, (ref, label) in enumerate(zip(references, decoded_labels))
        if ref != label
    ]

    print(f'{resource} mismatches before predict:', len(mismatches))

    if len(mismatches) > 0:
        print(mismatches[:5])
        raise ValueError(f'{resource}: raw references and prepared labels do not match')

    return test_dataset_raw_resource, test_dataset_resource



def predict_dataset_in_order(model, prepared_dataset, raw_dataset,
    data_collator, processor, batch_size=4):
    device = next(model.parameters()).device
    model.eval()

    rows = []

    for start in tqdm(range(0, len(prepared_dataset), batch_size)):
        end = min(start + batch_size, len(prepared_dataset))

        features = [prepared_dataset[i] for i in range(start, end)]

        batch = data_collator(features)

        input_batch = {key: value.to(device) for key, value in batch.items() if key != 'labels'}

        with torch.no_grad():
            logits = model(**input_batch).logits

        pred_ids = torch.argmax(logits, dim=-1)
        pred_str = processor.batch_decode(pred_ids)

        pred_str = [normalize_spaces(text) for text in pred_str]

        for local_i, prediction in enumerate(pred_str):
            raw_i = start + local_i
            raw_example = raw_dataset[raw_i]

            rows.append(
                {
                    'resource': raw_example['resource'],
                    'path': raw_example['path'],
                    'reference': normalize_spaces(raw_example['sentence']),
                    'prediction': prediction
                }
            )

    return pd.DataFrame(rows)


def evaluate_resource_test(resource, experiment_name, model, batch_size=4):
    test_dataset_raw_resource, test_dataset_resource = prepare_resource_test_dataset(resource)

    results_df = predict_dataset_in_order(
        model=model,
        prepared_dataset=test_dataset_resource,
        raw_dataset=test_dataset_raw_resource,
        data_collator=data_collator,
        processor=processor,
        batch_size=batch_size
    )

    wer = jiwer.wer(
        results_df['reference'].tolist(),
        results_df['prediction'].tolist()
    )

    cer = jiwer.cer(
        results_df['reference'].tolist(),
        results_df['prediction'].tolist()
    )

    predictions_path = f'mms_adapter_{experiment_name}_{resource}_test_predictions.csv'

    results_df.to_csv(predictions_path, index=False, encoding='utf-8-sig')

    print(f'{resource}_test WER:', wer)
    print(f'{resource}_test CER:', cer)

    row = {
        'model': 'MMS-1b-all adapter fine-tuning',
        'training': experiment_name,
        'subset': f'{resource}_test',
        'WER': wer,
        'CER': cer,
        'n_files': len(results_df),
        'predictions_file': predictions_path
    }

    return row, results_df


def get_best_dev_metrics(trainer, log_history_df):
    best_checkpoint = trainer.state.best_model_checkpoint
    best_dev_cer = trainer.state.best_metric

    best_step = None
    best_dev_loss = None
    best_dev_wer = None

    if best_checkpoint is not None:
        match = re.search(r'checkpoint-(\d+)', best_checkpoint)

        if match is not None:
            best_step = int(match.group(1))

            best_eval_rows = log_history_df[(log_history_df['step'] == best_step) &(log_history_df['eval_cer'].notna())]

            if len(best_eval_rows) > 0:
                best_eval_row = best_eval_rows.iloc[0]
                best_dev_loss = best_eval_row['eval_loss']
                best_dev_wer = best_eval_row['eval_wer']
                best_dev_cer = best_eval_row['eval_cer']

    return {
        'best_dev_checkpoint': best_checkpoint,
        'best_dev_step': best_step,
        'best_dev_loss': best_dev_loss,
        'best_dev_WER': best_dev_wer,
        'best_dev_CER': best_dev_cer
    }


def print_trainable_parameters(model):
    trainable_params = sum(
        param.numel()
        for param in model.parameters()
        if param.requires_grad
    )

    all_params = sum(
        param.numel()
        for param in model.parameters()
    )

    print(f'Trainable params: {trainable_params:,}')
    print(f'All params: {all_params:,}')
    print(f'Trainable share: {100 * trainable_params / all_params:.4f}%')


def save_adapter_model_and_processor(trainer, training_args, model_repo_id, processor, target_lang, hf_token):
    adapter_file = WAV2VEC2_ADAPTER_SAFE_FILE.format(target_lang)
    adapter_file = os.path.join(training_args.output_dir, adapter_file)

    adapter_weights = {name: tensor.detach().cpu() for name, tensor in trainer.model._get_adapters().items()}

    safe_save_file(adapter_weights, adapter_file, metadata={'format': 'pt'})

    print(adapter_file)

    processor.save_pretrained(training_args.output_dir)
    trainer.save_model(training_args.output_dir)

    trainer.push_to_hub()

    processor.push_to_hub(model_repo_id, private=False, token=hf_token)

In [14]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# Staged

## Stage 1: train on bible + radio

In [15]:
stage1_experiment_name = 'staged_bible_radio_to_chuklang_stage1'
stage1_model_repo_id = 'tadgeis/mms-1b-ckt-staged-bible-radio-to-chuklang-stage1'
resources_to_evaluate = ['chuklang', 'radio', 'bible']

create_repo(repo_id=stage1_model_repo_id, repo_type='model', private=False, exist_ok=True, token=hf_token)

processor.push_to_hub(stage1_model_repo_id, private=False, token=hf_token)

SEED = 42

stage1_train_dataset_raw = dataset_dict['train'].filter(lambda example: example['resource'] in ['bible', 'radio'])
stage1_eval_dataset_raw = dataset_dict['dev'].filter(lambda example: example['resource'] == 'chuklang')

stage1_train_dataset_raw = stage1_train_dataset_raw.shuffle(seed=SEED)

stage1_train_dataset = stage1_train_dataset_raw.map(prepare_dataset, remove_columns=stage1_train_dataset_raw.column_names, load_from_cache_file=False)
stage1_eval_dataset = stage1_eval_dataset_raw.map(prepare_dataset, remove_columns=stage1_eval_dataset_raw.column_names, load_from_cache_file=False)

print(stage1_train_dataset)
print(stage1_eval_dataset)

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Filter:   0%|          | 0/2714 [00:00<?, ? examples/s]

Filter:   0%|          | 0/140 [00:00<?, ? examples/s]

Map:   0%|          | 0/2055 [00:00<?, ? examples/s]

Map:   0%|          | 0/140 [00:00<?, ? examples/s]

Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 2055
})
Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 140
})


In [16]:
stage1_model = create_mms_adapter_model()
stage1_output_dir = stage1_model_repo_id.split('/')[-1]

stage1_training_args = TrainingArguments(
    output_dir=stage1_output_dir,

    group_by_length=True,

    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=16,

    eval_strategy='steps',
    save_strategy='steps',

    num_train_epochs=12,

    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),

    save_steps=50,
    eval_steps=50,
    logging_steps=50,

    learning_rate=1e-3,
    warmup_steps=25,

    save_total_limit=4,

    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,

    push_to_hub=True,
    hub_model_id=stage1_model_repo_id,
    hub_private_repo=False,
    hub_token=hf_token,
    hub_strategy='checkpoint',
    hub_always_push=True,

    report_to='none',
    disable_tqdm=False
)

stage1_trainer = Trainer(
    model=stage1_model,
    data_collator=data_collator,
    args=stage1_training_args,
    compute_metrics=compute_metrics,
    train_dataset=stage1_train_dataset,
    eval_dataset=stage1_eval_dataset,
    tokenizer=processor.feature_extractor,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.001
        )
    ]
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b-all and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([154]) in the checkpoint and torch.Size([42]) in the model instantiated
- lm_head.weight: found shape torch.Size([154, 1280]) in the checkpoint and torch.Size([42, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_1894/1074918229.py:45: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  stage1_trainer = Trainer(


In [17]:
print_trainable_parameters(stage1_model)

Trainable params: 2,204,970
All params: 964,702,378
Trainable share: 0.2286%


In [18]:
stage1_trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
50,6.512500,5.485976,1.000000,0.919151
100,3.444300,3.297744,1.000000,0.947869
150,1.134800,1.349748,0.938095,0.329248
200,0.523700,1.481469,0.957143,0.373880
250,0.471800,1.327716,0.946032,0.337114
300,0.453800,1.501656,0.958730,0.374611


TrainOutput(global_step=300, training_loss=2.0901587931315104, metrics={'train_runtime': 499.2075, 'train_samples_per_second': 49.398, 'train_steps_per_second': 3.101, 'total_flos': 4.259114569673952e+18, 'train_loss': 2.0901587931315104, 'epoch': 2.3255813953488373})

In [20]:
stage1_log_history_df = pd.DataFrame(stage1_trainer.state.log_history)

stage1_log_history_df.to_csv(f'mms_adapter_{stage1_experiment_name}_log_history.csv', index=False, encoding='utf-8-sig')

stage1_best_metrics = get_best_dev_metrics(stage1_trainer, stage1_log_history_df)

print(stage1_best_metrics)

save_adapter_model_and_processor(trainer=stage1_trainer, training_args=stage1_training_args,
    model_repo_id=stage1_model_repo_id, processor=processor, target_lang=target_lang, hf_token=hf_token)

{'best_dev_checkpoint': 'mms-1b-ckt-staged-bible-radio-to-chuklang-stage1/checkpoint-150', 'best_dev_step': 150, 'best_dev_loss': np.float64(1.3497482538223267), 'best_dev_WER': np.float64(0.9380952380952381), 'best_dev_CER': np.float64(0.32924821657216025)}
mms-1b-ckt-staged-bible-radio-to-chuklang-stage1/adapter.ckt.safetensors


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


In [21]:
del stage1_trainer
del stage1_model

gc.collect()
torch.cuda.empty_cache()

## Stage 2: train on chuklang

In [34]:
stage2_experiment_name = 'staged_bible_radio_to_chuklang_stage2'
stage2_model_repo_id = 'tadgeis/mms-1b-ckt-staged-bible-radio-to-chuklang-stage2'

create_repo(repo_id=stage2_model_repo_id, repo_type='model', private=False, exist_ok=True, token=hf_token)
processor.push_to_hub(stage2_model_repo_id, private=False, token=hf_token)

stage2_train_dataset_raw = dataset_dict['train'].filter(lambda example: example['resource'] == 'chuklang')
stage2_eval_dataset_raw = dataset_dict['dev'].filter(lambda example: example['resource'] == 'chuklang')

stage2_train_dataset_raw = stage2_train_dataset_raw.shuffle(seed=SEED)

stage2_train_dataset = stage2_train_dataset_raw.map(prepare_dataset,
    remove_columns=stage2_train_dataset_raw.column_names, load_from_cache_file=False)

stage2_eval_dataset = stage2_eval_dataset_raw.map(prepare_dataset,
    remove_columns=stage2_eval_dataset_raw.column_names, load_from_cache_file=False)

print(stage2_train_dataset)
print(stage2_eval_dataset)

No files have been modified since last commit. Skipping to prevent empty commit.


Map: 100%|##########| 659/659 [00:00<?, ? examples/s]

Map: 100%|##########| 140/140 [00:00<?, ? examples/s]

Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 659
})
Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 140
})


In [35]:
stage2_model = load_trained_mms_adapter_model(stage1_training_args.output_dir)
print_trainable_parameters(stage2_model)

stage2_output_dir = stage2_model_repo_id.split('/')[-1]

stage2_training_args = TrainingArguments(
    output_dir=stage2_output_dir,

    group_by_length=True,

    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=16,

    eval_strategy='steps',
    save_strategy='steps',

    num_train_epochs=36,

    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),

    save_steps=50,
    eval_steps=50,
    logging_steps=50,

    learning_rate=1e-3,
    warmup_steps=25,

    save_total_limit=4,

    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,

    push_to_hub=True,
    hub_model_id=stage2_model_repo_id,
    hub_private_repo=False,
    hub_token=hf_token,
    hub_strategy='checkpoint',
    hub_always_push=True,

    report_to='none',
    disable_tqdm=False
)

stage2_trainer = Trainer(
    model=stage2_model,
    data_collator=data_collator,
    args=stage2_training_args,
    compute_metrics=compute_metrics,
    train_dataset=stage2_train_dataset,
    eval_dataset=stage2_eval_dataset,
    tokenizer=processor.feature_extractor,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.001
        )
    ]
)

Trainable params: 2,204,970
All params: 964,702,378
Trainable share: 0.2286%


/tmp/ipykernel_1894/2394375064.py:47: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  stage2_trainer = Trainer(


In [36]:
stage2_trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
50,1.266900,0.917411,0.885714,0.233035
100,1.052700,0.824846,0.833333,0.212914
150,0.977400,0.778352,0.839683,0.207609
200,0.947800,0.747760,0.779365,0.189866
250,0.863800,0.736887,0.769841,0.187123
300,0.839100,0.698283,0.771429,0.179989
350,0.809800,0.703621,0.752381,0.176697
400,0.756400,0.706432,0.730159,0.176879
450,0.732100,0.694143,0.750794,0.174319
500,0.688000,0.699901,0.736508,0.175416


TrainOutput(global_step=900, training_loss=0.7647736740112304, metrics={'train_runtime': 907.4451, 'train_samples_per_second': 26.144, 'train_steps_per_second': 1.666, 'total_flos': 7.586237827840175e+18, 'train_loss': 0.7647736740112304, 'epoch': 21.428571428571427})

In [37]:
stage2_log_history_df = pd.DataFrame(stage2_trainer.state.log_history)

stage2_log_history_df.to_csv(f'mms_adapter_{stage2_experiment_name}_log_history.csv', index=False, encoding='utf-8-sig')

stage2_best_metrics = get_best_dev_metrics(stage2_trainer, stage2_log_history_df)

print(stage2_best_metrics)

save_adapter_model_and_processor(trainer=stage2_trainer, training_args=stage2_training_args,
    model_repo_id=stage2_model_repo_id, processor=processor, target_lang=target_lang, hf_token=hf_token)

{'best_dev_checkpoint': 'mms-1b-ckt-staged-bible-radio-to-chuklang-stage2/checkpoint-750', 'best_dev_step': 750, 'best_dev_loss': np.float64(0.672789990901947), 'best_dev_WER': np.float64(0.7126984126984127), 'best_dev_CER': np.float64(0.16462410828608012)}
mms-1b-ckt-staged-bible-radio-to-chuklang-stage2/adapter.ckt.safetensors


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


In [38]:
summary_rows = []
prediction_dfs = {}

for resource in resources_to_evaluate:
    row, results_df = evaluate_resource_test(
        resource=resource,
        experiment_name=stage2_experiment_name,
        model=stage2_trainer.model,
        batch_size=16
    )

    row['stage1_best_dev_checkpoint'] = stage1_best_metrics['best_dev_checkpoint']
    row['stage1_best_dev_step'] = stage1_best_metrics['best_dev_step']
    row['stage1_best_dev_loss'] = stage1_best_metrics['best_dev_loss']
    row['stage1_best_dev_WER'] = stage1_best_metrics['best_dev_WER']
    row['stage1_best_dev_CER'] = stage1_best_metrics['best_dev_CER']

    row['stage2_best_dev_checkpoint'] = stage2_best_metrics['best_dev_checkpoint']
    row['stage2_best_dev_step'] = stage2_best_metrics['best_dev_step']
    row['stage2_best_dev_loss'] = stage2_best_metrics['best_dev_loss']
    row['stage2_best_dev_WER'] = stage2_best_metrics['best_dev_WER']
    row['stage2_best_dev_CER'] = stage2_best_metrics['best_dev_CER']

    summary_rows.append(row)
    prediction_dfs[resource] = results_df

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    f'mms_adapter_{stage2_experiment_name}_results_summary.csv',
    index=False,
    encoding='utf-8-sig'
)

summary_df

Filter:   0%|          | 0/429 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

chuklang mismatches before predict: 0


  0%|          | 0/13 [00:00<?, ?it/s]

chuklang_test WER: 0.7332549941245593
chuklang_test CER: 0.18431589120530167


...

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

bible mismatches before predict: 0


  0%|          | 0/10 [00:00<?, ?it/s]

bible_test WER: 0.6794795978710823
bible_test CER: 0.11725498197646739


,model,training,subset,WER,CER,n_files,predictions_file,stage1_best_dev_checkpoint,stage1_best_dev_step,stage1_best_dev_loss,stage1_best_dev_WER,stage1_best_dev_CER,stage2_best_dev_checkpoint,stage2_best_dev_step,stage2_best_dev_loss,stage2_best_dev_WER,stage2_best_dev_CER
0,MMS-1b-all adapter fine-tuning,staged_bible_radio_to_chuklang_stage2,chuklang_test,0.733255,0.184316,200,mms_adapter_staged_bible_radio_to_chuklang_sta...,mms-1b-ckt-staged-bible-radio-to-chuklang-stag...,150,1.349748,0.938095,0.329248,mms-1b-ckt-staged-bible-radio-to-chuklang-stag...,750,0.67279,0.712698,0.164624
1,MMS-1b-all adapter fine-tuning,staged_bible_radio_to_chuklang_stage2,radio_test,0.887850,0.306621,83,mms_adapter_staged_bible_radio_to_chuklang_sta...,mms-1b-ckt-staged-bible-radio-to-chuklang-stag...,150,1.349748,0.938095,0.329248,mms-1b-ckt-staged-bible-radio-to-chuklang-stag...,750,0.67279,0.712698,0.164624
2,MMS-1b-all adapter fine-tuning,staged_bible_radio_to_chuklang_stage2,bible_test,0.679480,0.117255,146,mms_adapter_staged_bible_radio_to_chuklang_sta...,mms-1b-ckt-staged-bible-radio-to-chuklang-stag...,150,1.349748,0.938095,0.329248,mms-1b-ckt-staged-bible-radio-to-chuklang-stag...,750,0.67279,0.712698,0.164624


In [39]:
prediction_dfs['chuklang'].head()

,resource,path,reference,prediction
0,chuklang,A chatterbox and a wanton girl_2.wav,ӄоле итгъэт ӄынвэтэ ӈиръэ ӈэвысӄэтти элерэты н...,ӄоле итгъэтӄынвытэ ӈиръэ ӈэвысӄатэ лерэтынатанат
1,chuklang,A chatterbox and a wanton girl_3.wav,ӄол вэтгавӈавъым ӄол камэлгыӈав,ӄол вэтгавӈаъым ӄолкамэлгыӈа
2,chuklang,A chatterbox and a wanton girl_4.wav,ынкы илирыкы нантыӈӈонатъым,ынкы илирык нантыӈӈонатъыма
3,chuklang,Abramovich_4.wav,гэчевкы нынтыӄин таӈколё ынкы ныгынритӄин,эчевкын ынтыӄи таӈколёмкы ныгынритӄин
4,chuklang,An evil spirit and a dicky bird_1.wav,энмэн гатвален каԓьайӈын ынкъам пчеӄалгын,энмэ гатвален каԓьай ынкъам пчеӄалын


In [40]:
files_to_download = [
    Path(f'mms_adapter_{stage1_experiment_name}_log_history.csv'),
    Path(f'mms_adapter_{stage2_experiment_name}_log_history.csv'),
    Path(f'mms_adapter_{stage2_experiment_name}_results_summary.csv'),
]

for resource in resources_to_evaluate:
    files_to_download.append(
        Path(f'mms_adapter_{stage2_experiment_name}_{resource}_test_predictions.csv')
    )

for path in files_to_download:
    if path.exists():
        print(f'Ready to download: {path}')
    else:
        print(f'File not found: {path}')

Ready to download: mms_adapter_staged_bible_radio_to_chuklang_stage1_log_history.csv
Ready to download: mms_adapter_staged_bible_radio_to_chuklang_stage2_log_history.csv
Ready to download: mms_adapter_staged_bible_radio_to_chuklang_stage2_results_summary.csv
Ready to download: mms_adapter_staged_bible_radio_to_chuklang_stage2_chuklang_test_predictions.csv
Ready to download: mms_adapter_staged_bible_radio_to_chuklang_stage2_radio_test_predictions.csv
Ready to download: mms_adapter_staged_bible_radio_to_chuklang_stage2_bible_test_predictions.csv
